# Time-Series — โจทย์แบบ "ทำนายค่าในอนาคต" (Forecasting)

มีค่าตามเวลาในอดีต -> ทำนายค่าในอนาคต (เช่น ยอดขายพรุ่งนี้, ราคาสัปดาห์หน้า)

วิธีในโน้ตบุ๊กนี้ (มือใหม่):
- วิธีที่ 1 (ง่ายสุด) = สร้างฟีเจอร์จากวันที่ + ค่าย้อนหลัง (lag) แล้วใช้ `LightGBM` ทำนายเหมือน regression
- วิธีที่ 2 (ไม่บังคับ) = `AutoGluon TimeSeries` (เฉพาะทาง forecasting)


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อต้องทำนาย "ค่าตัวเลขในอนาคต" จากประวัติตามเวลา (มีคอลัมน์วันที่/เวลา)
ถ้าทำนาย "คลาสจากหน้าต่างสัญญาณ" -> ไปใช้ `signal_classification.ipynb`

ต้องแก้ (`# << แก้`): ชื่อ competition, `TIME_COL` (คอลัมน์วันที่), `TARGET`, `SERIES_COL` (ถ้ามีหลายซีรีส์)

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install lightgbm scikit-learn pandas numpy
!pip -q install -U "autogluon.timeseries"   # วิธีที่ 2 เท่านั้น

## ขั้นที่ 2 — เอาข้อมูลเข้า (เลือก 1 ใน 3 วิธี) + เช็ค GPU

เซลล์ล่างรองรับ 3 วิธี แก้แค่ตัวแปรบนสุดให้ตรงกับวิธีที่ใช้:

วิธี A (แนะนำ) ดาวน์โหลดจาก Kaggle อัตโนมัติ: ต้องมี `kaggle.json` (kaggle.com -> รูปโปรไฟล์ -> Settings -> Account -> Create New Token)
- บน `Kaggle Notebook`: ข้อมูลต่อไว้ที่ `/kaggle/input/...` แล้ว ไม่ต้องใส่ token
- บน `Colab/เครื่อง`: ใส่ `KAGGLE_USERNAME` + `KAGGLE_KEY`

วิธี B โหลดข้อมูลมาเครื่องตัวเองแล้วอัปขึ้น Colab เอง: ลากไฟล์ (เช่น `data.zip`) ไปวางในแถบ Files ซ้ายมือของ Colab
แล้วตั้ง `DATA_DIR = "/content"` (หรือโฟลเดอร์ที่วาง) -> เซลล์จะแตก zip ให้เอง ไม่ต้องใช้ token

วิธี C ต่อ Google Drive: รัน `from google.colab import drive; drive.mount('/content/drive')` ก่อน แล้วตั้ง `DATA_DIR = "/content/drive/MyDrive/โฟลเดอร์ของคุณ"`

เซลล์นี้เช็ค GPU ให้ด้วย ถ้างานเป็น deep learning (รูป/ข้อความ/สัญญาณดิบ) ควรเปิด GPU: เมนู `Runtime > Change runtime type > T4 GPU`

In [ ]:
import os, glob, subprocess

# ----- วิธี A: ดาวน์โหลดจาก Kaggle -----
COMP = "ใส่-slug-ของ-competition-forecasting-ตรงนี้"      # << แก้: slug ท้าย URL เช่น kaggle.com/competitions/titanic -> "titanic"  (ใช้เฉพาะวิธี A)
KAGGLE_USERNAME = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "peetwan1"  (บน Kaggle Notebook เว้นว่างได้)
KAGGLE_KEY      = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "0a1b2c3d4e5f..." (คีย์ยาว ~32 ตัวจาก kaggle.json)
# ----- วิธี B/C: อัปโหลดเอง / ต่อ Google Drive -----
DATA_DIR = ""          # << แก้: ถ้าอัปโหลดข้อมูลเอง ใส่ path โฟลเดอร์ เช่น "/content" (วิธี B) หรือ "/content/drive/MyDrive/data" (วิธี C) ; ใช้ Kaggle (วิธี A) ปล่อยว่าง

# เช็ค GPU (ถ้าไม่มี + งานเป็น deep learning -> เปิดที่เมนู Runtime > Change runtime type > T4 GPU)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี (งาน deep learning จะช้า; งานตาราง/ตัดคำ ไม่ต้องใช้ GPU ก็ได้)")
except Exception:
    pass

def get_data(comp):
    # วิธี B/C: ผู้ใช้ตั้ง DATA_DIR เอง (อัปโหลด/ต่อ Drive) -- แตก zip ในโฟลเดอร์นั้นให้ด้วยถ้ามี
    if DATA_DIR and os.path.isdir(DATA_DIR):
        for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
            subprocess.run(["unzip", "-o", "-q", z, "-d", DATA_DIR])
        print("ใช้ข้อมูลจากโฟลเดอร์ที่ตั้งเอง:", DATA_DIR); return DATA_DIR
    # บน Kaggle Notebook ข้อมูลต่อไว้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle ข้อมูลอยู่ที่", kpath); return kpath
    # วิธี A: ดาวน์โหลดด้วย Kaggle API
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("\nไฟล์ที่มี (ดูชื่อไฟล์/โฟลเดอร์ แล้วแก้ในเซลล์ถัดไปให้ตรง):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — โหลดข้อมูล + CONFIG

In [ ]:
import os, pandas as pd, numpy as np
TRAIN_CSV=os.path.join(DATA_DIR,"train.csv"); TEST_CSV=os.path.join(DATA_DIR,"test.csv")
SAMPLE_SUB=os.path.join(DATA_DIR,"sample_submission.csv")
train=pd.read_csv(TRAIN_CSV); test=pd.read_csv(TEST_CSV); sample=pd.read_csv(SAMPLE_SUB)
TIME_COL  = "date"      # << แก้: ชื่อคอลัมน์วันที่/เวลา -- ดูจาก list(train.columns) เช่น "date", "timestamp", "datetime"
TARGET    = "sales"     # << แก้: ชื่อคอลัมน์ค่าที่ทำนาย "ในไฟล์ train" -- ดูจาก train.head() เช่น "sales", "value", "y"
SUB_TARGET= sample.columns[1]   # ชื่อคอลัมน์คำตอบ "ในไฟล์ submission" (มักไม่ต้องแก้ -- ชื่ออาจไม่เหมือน TARGET ของ train)
SERIES_COL= None        # << แก้: ถ้ามีหลายซีรีส์ใส่ชื่อคอลัมน์ เช่น "store_id", ถ้าซีรีส์เดียวใส่ None
ID_COL=sample.columns[0]
print("คอลัมน์:",list(train.columns)); display(train.head()); display(sample.head())

## 🔎 โจทย์นี้ต้องส่งอะไร + วัดด้วยอะไร (รันก่อนทำ submission)

เซลล์นี้ตอบ 3 คำถามที่มือใหม่มักไม่รู้:
- ต้องเติม "คอลัมน์อะไร" ลง submission และเป็น "ชนิดไหน" (ดูจาก sample_submission)
- โจทย์วัดด้วย "metric อะไร" (ดึงจาก Kaggle ให้อัตโนมัติ)
- ต้องส่งเป็น "ป้าย / ความน่าจะเป็น / ตัวเลข" แบบไหน

In [ ]:
# บอกว่าต้องเติมอะไรลง submission + ตัวอย่างค่าที่ sample ให้มา
print("ไฟล์ส่งต้องมีคอลัมน์:", list(sample.columns), "| จำนวนแถว:", len(sample))
for _c in list(sample.columns)[1:]:
    print(f"  - เติม '{_c}': ชนิด {sample[_c].dtype}, ตัวอย่างค่าใน sample = {list(sample[_c].dropna().unique()[:3])}")
# ดึง metric จาก Kaggle อัตโนมัติ (ถ้าต่อ API ได้)
_metric = None
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    _api = KaggleApi(); _api.authenticate()
    _resp = _api.competitions_list(search=COMP)
    for _co in (getattr(_resp, "competitions", _resp) or []):   # kaggle ใหม่คืน response (ใช้ .competitions), เก่าคืน list
        if str(getattr(_co, "ref", "")).rstrip("/").split("/")[-1] == COMP:
            _metric = getattr(_co, "evaluation_metric", None) or getattr(_co, "evaluationMetric", None); break
except Exception:
    pass
def _how_to_send(m):
    m = (m or "").lower()
    if any(k in m for k in ["auc","roc","log loss","logloss","brier","probab"]): return "ความน่าจะเป็น (เลข 0-1)"
    if any(k in m for k in ["rmse","mae","mse","r2","rmsle","error","mape","smape"]): return "ตัวเลขต่อเนื่อง"
    return "ป้าย/คลาส (เช่น 0/1 หรือชื่อคลาส)"
print("\nMetric (ดึงจาก Kaggle):", _metric or "ดึงไม่ได้ -> เปิด tab Evaluation บนหน้า Kaggle อ่านเอง")
print("=> ปกติต้องส่งเป็น:", _how_to_send(_metric), "(เช็คกับ tab Evaluation อีกที)")

## วิธีที่ 1 — ฟีเจอร์วันที่ + lag + LightGBM (ง่ายสุด ใช้ได้จริง)

หลักคิด: เปลี่ยนปัญหา forecasting ให้เป็น regression ธรรมดา โดยทำฟีเจอร์จากวันที่ + ค่าย้อนหลัง (lag) แล้วให้ LightGBM เรียนรู้

จุดที่มือใหม่พลาดบ่อยสุด (อ่านให้เข้าใจ):
- `lag` = ค่าย้อนหลัง เช่น `lag_1` = ยอดของเมื่อวาน, `lag_7` = ยอดของ 7 วันก่อน (เป็นฟีเจอร์ที่สำคัญที่สุด)
- ตอนเทรนเรามีค่าจริงในอดีตให้ทำ lag ได้ แต่ตอนทำนายอนาคต (test) เรา "ยังไม่รู้" ค่าอนาคต
- ถ้าปล่อย lag ของ test เป็นค่าว่าง (NaN) โมเดลจะเดามั่ว คะแนนพังแบบเงียบ ๆ (ไม่ error แต่ได้คะแนนแย่)
- วิธีที่ถูก: ทำนายทีละแถวเรียงตามเวลา พอได้ค่าทำนายแล้วเอาไปเป็น "ประวัติ" ของแถวถัดไป (เรียกว่า recursive forecasting)

In [ ]:
import lightgbm as lgb
LAGS = (1,7,14,28)   # << แก้ค่า lag ให้เข้ากับความถี่ข้อมูล: รายวัน -> (1,7,14,28); รายชั่วโมง -> (1,24,168); รายเดือน -> (1,12)
def add_time_features(df):
    df=df.copy(); df[TIME_COL]=pd.to_datetime(df[TIME_COL])
    df["year"]=df[TIME_COL].dt.year; df["month"]=df[TIME_COL].dt.month
    df["day"]=df[TIME_COL].dt.day; df["dow"]=df[TIME_COL].dt.dayofweek
    df["doy"]=df[TIME_COL].dt.dayofyear
    return df
def add_lags(df):
    df=df.sort_values(TIME_COL).copy()
    g=df.groupby(SERIES_COL)[TARGET] if SERIES_COL else df[TARGET]
    for L in LAGS:
        df[f"lag_{L}"]= (g.shift(L) if SERIES_COL else df[TARGET].shift(L))
    return df

# 1) เทรนบนข้อมูล train ที่มี lag จริง
tr=add_lags(add_time_features(train)).dropna()
feat_cols=[c for c in tr.columns if c not in [TARGET,TIME_COL,ID_COL] and pd.api.types.is_numeric_dtype(tr[c])]
m=lgb.LGBMRegressor(n_estimators=2000,learning_rate=0.02,num_leaves=63,random_state=42,verbose=-1)
m.fit(tr[feat_cols], tr[TARGET])

# 2) ทำนาย test แบบ recursive: เติม lag จากประวัติจริง แล้วใช้ค่าที่ทำนายเป็นประวัติของแถวถัดไป
#    (ตัวอย่างนี้ทำกรณีซีรีส์เดียว; ถ้ามีหลายซีรีส์ต้องวนแยกตาม SERIES_COL)
test2=add_time_features(test).sort_values(TIME_COL).reset_index(drop=True)
series=list(add_time_features(train).sort_values(TIME_COL)[TARGET].values)  # ประวัติค่าจริงจาก train
preds=[]
for _,row in test2.iterrows():
    fr={c:(row[c] if c in test2.columns else np.nan) for c in feat_cols}  # เอาค่าฟีเจอร์อื่น ๆ จากแถว test ด้วย (กัน KeyError ถ้ามีฟีเจอร์เพิ่ม)
    for L in LAGS:
        fr[f"lag_{L}"]= series[-L] if len(series)>=L else np.nan
    yhat=float(m.predict(pd.DataFrame([fr])[feat_cols])[0])
    preds.append(yhat); series.append(yhat)   # ใช้ค่าที่ทำนายต่อเป็นประวัติ
out=sample.copy(); out[SUB_TARGET]=preds
out.to_csv("submission.csv",index=False); print("บันทึก submission.csv"); display(out.head())

## วิธีที่ 2 — AutoGluon TimeSeries (ไม่บังคับ เฉพาะทาง forecasting)

เหมาะเมื่อข้อมูลเป็นรูปแบบ long (item_id, timestamp, target) ชัดเจน

In [ ]:
# from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
# tsdf = TimeSeriesDataFrame.from_data_frame(train, id_column=SERIES_COL, timestamp_column=TIME_COL)
# predictor = TimeSeriesPredictor(target=TARGET, prediction_length=28).fit(tsdf, time_limit=600)
# pred = predictor.predict(tsdf)   # แล้วแมปกลับเข้า sample_submission ตามรูปแบบจริง
print("ปลดคอมเมนต์เซลล์นี้ถ้าข้อมูลเป็นรูปแบบ long ชัดเจน")

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
import pandas as pd, glob, os
FILE = "submission.csv"   # << แก้เป็นไฟล์ที่จะส่ง เช่น "submission_lgbm.csv" หรือ "submission_timm.csv"
# ตรวจรูปแบบก่อนส่งอัตโนมัติ (กันแก้ config ผิดแล้วส่งฟอร์แมตเพี้ยน -> ได้ 0 คะแนน)
_sub = pd.read_csv(FILE)
_sam = glob.glob(os.path.join(DATA_DIR, "*ample*ubmission*.csv"))
if _sam:
    _s = pd.read_csv(_sam[0])
    print("คอลัมน์ตรง sample:", list(_sub.columns)==list(_s.columns), "| จำนวนแถวตรง:", len(_sub)==len(_s))
    assert list(_sub.columns)==list(_s.columns), f"คอลัมน์ไม่ตรง sample! {list(_sub.columns)} != {list(_s.columns)} -> แก้ก่อนส่ง"
print("พร้อมส่ง:", FILE, _sub.shape)
!kaggle competitions submit -c {COMP} -f {FILE} -m "lightgbm forecasting"
!kaggle competitions submissions -c {COMP} | head